In [ ]:
# Example 1: Basic Catalog and Table Schema Operations
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import *
import tempfile
import os

# Create a local catalog configuration for demonstration
catalog_config = {
    "type": "sql",
    "uri": f"sqlite:///{tempfile.mktemp()}.db",
}

# Load catalog and explore basic operations
try:
    catalog = load_catalog("demo_catalog", **catalog_config)
    
    # Define a sample schema for analytics data
    schema = Schema(
        NestedField(field_id=1, name="event_id", field_type=StringType(), required=True),
        NestedField(field_id=2, name="user_id", field_type=LongType(), required=True),
        NestedField(field_id=3, name="timestamp", field_type=TimestampType(), required=True),
        NestedField(field_id=4, name="event_type", field_type=StringType(), required=True),
        NestedField(field_id=5, name="properties", field_type=MapType(
            key_id=6, key_type=StringType(),
            value_id=7, value_type=StringType(), 
            value_required=False), required=False)
    )
    
    print("Catalog loaded successfully")
    print(f"Schema created with {len(schema.fields)} fields:")
    for field in schema.fields:
        print(f"  {field.name}: {field.field_type} (required: {field.required})")
    
    # List available namespaces
    namespaces = list(catalog.list_namespaces())
    print(f"Available namespaces: {namespaces}")
    
    # Create a namespace if none exist
    if not namespaces:
        catalog.create_namespace("analytics")
        print("Created 'analytics' namespace")
        
except Exception as e:
    print(f"Basic catalog operations completed with note: {str(e)[:100]}")
    print("Schema and type system demonstrated successfully")
    raise AssertionError(
        f"pyiceberg example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

In [ ]:
# Example 2: Table Creation and Partitioning Strategies
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import BucketTransform, DayTransform
from pyiceberg.types import *
from pyiceberg.schema import Schema
import tempfile

# Define table schema for partitioned analytics data
analytics_schema = Schema(
    NestedField(field_id=1, name="event_id", field_type=StringType(), required=True),
    NestedField(field_id=2, name="user_id", field_type=LongType(), required=True),
    NestedField(field_id=3, name="event_date", field_type=DateType(), required=True),
    NestedField(field_id=4, name="region", field_type=StringType(), required=True),
    NestedField(field_id=5, name="revenue", field_type=DecimalType(10, 2), required=False),
    NestedField(field_id=6, name="device_type", field_type=StringType(), required=False)
)

# Create partitioning strategy for efficient querying
partition_spec = PartitionSpec(
    PartitionField(source_id=3, field_id=1000, transform=DayTransform(), name="event_day"),
    PartitionField(source_id=4, field_id=1001, transform=BucketTransform(16), name="region_bucket"),
    spec_id=0
)

print("Analytics table schema:")
for field in analytics_schema.fields:
    print(f"  {field.name}: {field.field_type}")

print(f"\nPartitioning strategy with {len(partition_spec.fields)} partitions:")
for partition_field in partition_spec.fields:
    print(f"  {partition_field.name}: transform={partition_field.transform}")

# Demonstrate schema evolution capabilities
# Create an evolved schema with additional field
evolved_schema = Schema(
    *analytics_schema.fields,
    NestedField(field_id=7, name="session_duration", field_type=IntegerType(), required=False)
)
print(f"\nEvolved schema now has {len(evolved_schema.fields)} fields")
print(f"New field: {evolved_schema.fields[-1].name}")

# Create table properties for optimization
table_properties = {
    "write.format.default": "parquet",
    "write.parquet.compression-codec": "snappy",
    "write.metadata.compression-codec": "gzip"
}

print(f"\nTable properties configured:")
for key, value in table_properties.items():
    print(f"  {key}: {value}")
    
print("Table creation and partitioning strategy defined successfully")

In [ ]:
# Example 3: Data Types and Schema Evolution  
from pyiceberg.types import *
from pyiceberg.schema import Schema
from pyiceberg.catalog import load_catalog
import tempfile

# Demonstrate comprehensive data type support
complex_schema = Schema(
    # Primitive types
    NestedField(1, "id", LongType(), required=True),
    NestedField(2, "name", StringType(), required=True),
    NestedField(3, "price", DecimalType(precision=10, scale=2), required=True),
    NestedField(4, "is_active", BooleanType(), required=False),
    NestedField(5, "created_at", TimestampType(), required=True),
    NestedField(6, "updated_date", DateType(), required=False),
    NestedField(7, "data_blob", BinaryType(), required=False),
    
    # Complex types
    NestedField(8, "tags", ListType(element_id=9, element_type=StringType(), 
                                   element_required=False), required=False),
    NestedField(10, "metadata", MapType(key_id=11, key_type=StringType(),
                                       value_id=12, value_type=StringType(),
                                       value_required=False), required=False),
    NestedField(13, "coordinates", StructType(
        NestedField(14, "lat", DoubleType(), required=True),
        NestedField(15, "lon", DoubleType(), required=True),
        NestedField(16, "elevation", FloatType(), required=False)
    ), required=False)
)

print("Complex schema with various data types:")
for field in complex_schema.fields:
    print(f"  {field.name}: {field.field_type} ({'required' if field.required else 'optional'})")

# Demonstrate schema evolution operations
print("\nSchema evolution capabilities:")

# Add new column by creating evolved schema
evolved_schema_1 = Schema(
    *complex_schema.fields,
    NestedField(17, "category", StringType(), required=False)
)
print(f"Added 'category' column, now {len(evolved_schema_1.fields)} fields")

# Add nested column by creating new schema with updated struct
coordinates_with_accuracy = StructType(
    NestedField(14, "lat", DoubleType(), required=True),
    NestedField(15, "lon", DoubleType(), required=True),
    NestedField(16, "elevation", FloatType(), required=False),
    NestedField(18, "accuracy", IntegerType(), required=False)
)
evolved_schema_2 = Schema(
    *[f for f in evolved_schema_1.fields if f.name != 'coordinates'],
    NestedField(13, "coordinates", coordinates_with_accuracy, required=False)
)
print(f"Added nested 'coordinates.accuracy', schema updated")

# Schema validation and field access
try:
    field_by_name = evolved_schema_2.find_field("coordinates.lat")
    print(f"Found nested field: {field_by_name.name} with type {field_by_name.field_type}")
    
    field_by_id = evolved_schema_2.find_field(1)  
    print(f"Found field by ID: {field_by_id.name}")
    
except Exception as e:
    print(f"Schema field access: {e}")
    raise

print("Schema evolution and data type validation completed")

In [ ]:
# Example 4: Table Metadata and Transaction Management
from pyiceberg.table import TableMetadata, Table
from pyiceberg.types import *
from pyiceberg.schema import Schema
import tempfile
import json
from datetime import datetime

# Create sample table metadata for demonstration
metadata_schema = Schema(
    NestedField(1, "transaction_id", StringType(), required=True),
    NestedField(2, "amount", DecimalType(10, 2), required=True),
    NestedField(3, "timestamp", TimestampType(), required=True)
)

# Simulate table metadata operations
print("Table Metadata Operations:")

# Create mock metadata structure
metadata_info = {
    "format-version": 2,
    "table-uuid": "550e8400-e29b-41d4-a716-446655440000",
    "location": f"file://{tempfile.mkdtemp()}/transactions",
    "last-sequence-number": 0,
    "last-updated-ms": int(datetime.now().timestamp() * 1000),
    "last-column-id": 3,
    "current-schema-id": 0,
    "schemas": [{"schema-id": 0, "fields": []}],
    "default-spec-id": 0,
    "partition-specs": [{"spec-id": 0, "fields": []}],
    "last-partition-id": 999,
    "default-sort-order-id": 0,
    "sort-orders": [{"order-id": 0, "fields": []}],
    "properties": {},
    "current-snapshot-id": -1,
    "refs": {},
    "snapshots": [],
    "snapshot-log": [],
    "metadata-log": []
}

print(f"Table UUID: {metadata_info['table-uuid']}")
print(f"Format version: {metadata_info['format-version']}")
print(f"Last updated: {datetime.fromtimestamp(metadata_info['last-updated-ms'] / 1000)}")
print(f"Current schema ID: {metadata_info['current-schema-id']}")

# Demonstrate transaction tracking
transaction_log = []
operations = ["CREATE_TABLE", "ADD_COLUMN", "UPDATE_PARTITION", "ADD_SNAPSHOT"]

for i, operation in enumerate(operations):
    transaction_entry = {
        "sequence_number": i + 1,
        "operation": operation,
        "timestamp": int(datetime.now().timestamp() * 1000),
        "snapshot_id": 1000 + i if operation == "ADD_SNAPSHOT" else None
    }
    transaction_log.append(transaction_entry)

print(f"\nTransaction Log ({len(transaction_log)} operations):")
for entry in transaction_log:
    snap_info = f" (snapshot: {entry['snapshot_id']})" if entry['snapshot_id'] else ""
    print(f"  {entry['sequence_number']}: {entry['operation']}{snap_info}")

print("Table metadata and transaction management demonstrated")

In [ ]:
# Example 5: Production Analytics Pipeline with PyIceberg
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema  
from pyiceberg.types import *
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import HourTransform, BucketTransform
import tempfile
import json
from datetime import datetime, timedelta

# Production analytics table design
analytics_pipeline_schema = Schema(
    NestedField(1, "event_id", UUIDType(), required=True),
    NestedField(2, "user_id", LongType(), required=True),
    NestedField(3, "session_id", StringType(), required=True),
    NestedField(4, "event_timestamp", TimestampType(), required=True),
    NestedField(5, "event_name", StringType(), required=True),
    NestedField(6, "platform", StringType(), required=True),
    NestedField(7, "country_code", StringType(), required=True),
    NestedField(8, "revenue_usd", DecimalType(12, 2), required=False),
    NestedField(9, "properties", MapType(
        key_id=10, key_type=StringType(),
        value_id=11, value_type=StringType(),
        value_required=False), required=False)
)

# Production partitioning strategy for performance
partition_strategy = PartitionSpec(
    PartitionField(source_id=4, field_id=1000, transform=HourTransform(), name="event_hour"),
    PartitionField(source_id=6, field_id=1001, transform=BucketTransform(8), name="platform_bucket"),
    PartitionField(source_id=7, field_id=1002, transform=BucketTransform(32), name="country_bucket"),
    spec_id=1
)

print("Production Analytics Pipeline Configuration:")
print(f"Schema: {len(analytics_pipeline_schema.fields)} fields optimized for analytics")
print(f"Partitioning: {len(partition_strategy.fields)} partition levels for performance")

# Table optimization properties for production
optimization_properties = {
    "write.format.default": "parquet",
    "write.parquet.compression-codec": "zstd",
    "write.parquet.page-size-bytes": "1048576",
    "write.parquet.row-group-size-bytes": "134217728",
    "write.target-file-size-bytes": "536870912",
    "read.parquet.vectorization.enabled": "true",
    "history.expire.max-snapshot-age-ms": str(7 * 24 * 60 * 60 * 1000),  # 7 days
}

print(f"\nOptimization Properties ({len(optimization_properties)} configurations):")
for prop, value in optimization_properties.items():
    print(f"  {prop}: {value}")

# Simulate data quality and monitoring setup
data_quality_checks = [
    {"field": "event_timestamp", "check": "not_null", "critical": True},
    {"field": "user_id", "check": "positive", "critical": True},
    {"field": "event_name", "check": "allowed_values", "values": ["page_view", "click", "purchase"]},
    {"field": "revenue_usd", "check": "range", "min": 0, "max": 10000},
    {"field": "country_code", "check": "format", "pattern": "[A-Z]{2}"}
]

print(f"\nData Quality Framework ({len(data_quality_checks)} checks):")
for check in data_quality_checks:
    critical = " (CRITICAL)" if check.get("critical") else ""
    print(f"  {check['field']}: {check['check']}{critical}")

# Performance monitoring metrics
performance_metrics = {
    "daily_row_count": 50_000_000,
    "avg_query_time_ms": 2500,
    "storage_gb_per_day": 12.5,
    "partition_count_per_day": 192,  # 24 hours * 8 platform buckets
    "compression_ratio": 4.2
}

print(f"\nExpected Performance Metrics:")
for metric, value in performance_metrics.items():
    print(f"  {metric}: {value:,}")

print("\nProduction analytics pipeline configuration completed")